# OPENBREWERY - SILVER NOTEBOOK

# 1. GENERAL SETTINGS

## 1.1 Load Variables

In [0]:
%run ./00-Config

## 1.2 Imports

In [0]:
from delta.tables import DeltaTable

from pyspark.sql import functions as F

from pyspark.sql.window import Window

from pyspark.sql.types import *

from datetime import datetime

## 1.3 Notebook Variables

In [0]:
silver_table = (
    f"{catalog_name}.silver.openbrewery_silver"
)

gold_control_table = (
    f"{catalog_name}.control.gold_control"
)

monitoring_table = (
    f"{catalog_name}.control.pipeline_monitoring"
)


data_quality_table = (
    f"{catalog_name}.control.data_quality"
)

pipeline_name = "openbrewery_gold"

layer = "gold"

execution_id = datetime.utcnow().strftime(
    "%Y%m%d%H%M%S"
)

start_time = datetime.utcnow()

# ==========================================
# Dimensions
# ==========================================

dim_brewery_type_table = (
    f"{catalog_name}.gold.dim_brewery_type"
)

dim_location_table = (
    f"{catalog_name}.gold.dim_location"
)

dim_brewery_table = (
    f"{catalog_name}.gold.dim_brewery"
)

# ==========================================
# Fact
# ==========================================

fact_brewery_table = (
    f"{catalog_name}.gold.fact_brewery"

## 1.4 Get Last Timestamp

In [0]:
control_df = spark.sql(f"""

SELECT
    MAX(last_ingestion_timestamp) AS last_ts

FROM {gold_control_table}

WHERE pipeline_name = 'gold_layer'

""")

last_ts = control_df.collect()[0]["last_ts"]

print("=" * 60)
print(f"Último timestamp GOLD: {last_ts}")
print("=" * 60)

# 2. Execution

## 2.1 Silver Reading

In [0]:
silver_df = (

    spark.read.table(silver_table)

    .filter(
        F.col("is_current") == True
    )
)

## 2.2 Incremental Processing

In [0]:
if last_ts:

    silver_df = (

        silver_df

        .filter(
            F.col("ingestion_timestamp") > F.lit(last_ts)
        )
    )

## 2.3 Standardizations

In [0]:
silver_df = (

    silver_df

    .withColumn(
        "name",
        F.trim(F.col("name"))
    )

    .withColumn(
        "brewery_type",
        F.lower(F.col("brewery_type"))
    )

    .withColumn(
        "city",
        F.trim(F.col("city"))
    )

    .withColumn(
        "state",
        F.trim(F.col("state"))
    )

    .withColumn(
        "country",
        F.upper(F.col("country"))
    )
)

## 2.4 Remove Duplicates

In [0]:
silver_df = silver_df.dropDuplicates(["id"])

## 2.5 Dim Brewery Type

In [0]:
dim_brewery_type_df = (

    silver_df

    .select("brewery_type")

    .filter(
        F.col("brewery_type").isNotNull()
    )

    .dropDuplicates()

    .withColumn(

        "brewery_type_sk",

        F.sha2(
            F.col("brewery_type"),
            256
        )
    )
)

## 2.5 Dim Brewery Type Data Quality

In [0]:
duplicate_brewery_type_sk = (

    dim_brewery_type_df

    .groupBy("brewery_type_sk")

    .count()

    .filter(F.col("count") > 1)

    .count()
)

## 2.5 Merge Dim Brewery Type

In [0]:
if not spark.catalog.tableExists(dim_brewery_type_table):

    (
        dim_brewery_type_df.write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(dim_brewery_type_table)
    )

else:

    delta_dim_type = DeltaTable.forName(
        spark,
        dim_brewery_type_table
    )

    (
        delta_dim_type.alias("target")

        .merge(

            dim_brewery_type_df.alias("source"),

            """
            target.brewery_type_sk =
            source.brewery_type_sk
            """
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

## 2.6 Dim Location

In [0]:
dim_location_df = (

    silver_df

    .select(
        "city",
        "state",
        "country"
    )

    .filter(
        F.col("country").isNotNull()
    )

    .dropDuplicates()

    .withColumn(

        "location_sk",

        F.sha2(

            F.concat_ws(
                "||",
                F.col("city"),
                F.col("state"),
                F.col("country")
            ),

            256
        )
    )
)

## 2.6 Dim Location Data Quality

In [0]:
duplicate_location_sk = (

    dim_location_df

    .groupBy("location_sk")

    .count()

    .filter(F.col("count") > 1)

    .count()
)

## 2.7 Merge Dim Location

In [0]:
if not spark.catalog.tableExists(dim_location_table):

    (
        dim_location_df.write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(dim_location_table)
    )

else:

    delta_dim_location = DeltaTable.forName(
        spark,
        dim_location_table
    )

    (
        delta_dim_location.alias("target")

        .merge(

            dim_location_df.alias("source"),

            """
            target.location_sk =
            source.location_sk
            """
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

## 2.7 Dim Brewery

In [0]:
dim_brewery_df = (

    silver_df

    .withColumn(

        "brewery_sk",

        F.sha2(
            F.col("id"),
            256
        )
    )

    .withColumn(

        "brewery_type_sk",

        F.sha2(
            F.col("brewery_type"),
            256
        )
    )

    .withColumn(

        "location_sk",

        F.sha2(

            F.concat_ws(
                "||",
                F.col("city"),
                F.col("state"),
                F.col("country")
            ),

            256
        )
    )

    .select(

        "brewery_sk",

        F.col("id").alias("brewery_id"),

        F.col("name").alias("brewery_name"),

        "brewery_type_sk",

        "location_sk",

        "website_url",

        "phone",

        "ingestion_timestamp"
    )
)

## 2.8 Dim Brewery Data Quality

In [0]:
duplicate_brewery_sk = (

    dim_brewery_df

    .groupBy("brewery_sk")

    .count()

    .filter(F.col("count") > 1)

    .count()
)

## 2.9 Merge Dim Brewery

In [0]:
if not spark.catalog.tableExists(dim_brewery_table):

    (
        dim_brewery_df.write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(dim_brewery_table)
    )

else:

    delta_dim_brewery = DeltaTable.forName(
        spark,
        dim_brewery_table
    )

    (
        delta_dim_brewery.alias("target")

        .merge(

            dim_brewery_df.alias("source"),

            """
            target.brewery_sk =
            source.brewery_sk
            """
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )

## 2.9 Fact Brewery

In [0]:
fact_brewery_df = (

    silver_df

    .withColumn(

        "snapshot_date",

        F.current_date()
    )

    .withColumn(

        "brewery_sk",

        F.sha2(
            F.col("id"),
            256
        )
    )

    .withColumn(

        "brewery_type_sk",

        F.sha2(
            F.col("brewery_type"),
            256
        )
    )

    .withColumn(

        "location_sk",

        F.sha2(

            F.concat_ws(
                "||",
                F.col("city"),
                F.col("state"),
                F.col("country")
            ),

            256
        )
    )

    .withColumn(
        "brewery_count",
        F.lit(1)
    )

    .select(

        "snapshot_date",

        "brewery_sk",

        "brewery_type_sk",

        "location_sk",

        "brewery_count",

        "ingestion_timestamp"
    )
)

## 2.10 Fact Brewery Data Quality

In [0]:
duplicate_fact_records = (

    fact_brewery_df

    .groupBy(
        "snapshot_date",
        "brewery_sk"
    )

    .count()

    .filter(F.col("count") > 1)

    .count()
)

null_surrogate_keys = (

    fact_brewery_df

    .filter(

        F.col("brewery_sk").isNull()

        |

        F.col("brewery_type_sk").isNull()

        |

        F.col("location_sk").isNull()
    )

    .count()
)

negative_brewery_count = (

    fact_brewery_df

    .filter(
        F.col("brewery_count") < 0
    )

    .count()
)

## 2.10 Remove Duplicates Fact

In [0]:
# COMMAND ----------
# ==========================================
# REMOVE DUPLICADOS FACT
# ==========================================

fact_brewery_df = (

    fact_brewery_df

    .dropDuplicates([
        "snapshot_date",
        "brewery_sk"
    ])
)

## 2.11 Fact Validation

In [0]:
fact_brewery_df = (

    fact_brewery_df

    .filter(
        F.col("brewery_count") >= 0
    )
)

## 2.12 Data Quality Orphaned Foreign Keys

In [0]:
orphan_brewery_type_sk = (

    fact_brewery_df.alias("fact")

    .join(
        dim_brewery_type_df.alias("dim"),
        on="brewery_type_sk",
        how="leftanti"
    )

    .count()
)

orphan_location_sk = (

    fact_brewery_df.alias("fact")

    .join(
        dim_location_df.alias("dim"),
        on="location_sk",
        how="leftanti"
    )

    .count()
)

orphan_brewery_sk = (

    fact_brewery_df.alias("fact")

    .join(
        dim_brewery_df.alias("dim"),
        on="brewery_sk",
        how="leftanti"
    )

    .count()
)

## 2.13 Fact Reading

In [0]:
if not spark.catalog.tableExists(fact_brewery_table):

    (
        fact_brewery_df.write

        .format("delta")

        .partitionBy("snapshot_date")

        .mode("overwrite")

        .saveAsTable(fact_brewery_table)
    )

else:

    delta_fact = DeltaTable.forName(
        spark,
        fact_brewery_table
    )

    (
        delta_fact.alias("target")

        .merge(

            fact_brewery_df.alias("source"),

            """
            target.snapshot_date = source.snapshot_date
            AND
            target.brewery_sk = source.brewery_sk
            """
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

## 2.14 Data Quality Logging

In [0]:
dq_data = [

    (
        execution_id,
        pipeline_name,
        layer,
        "dim_brewery_type",
        "duplicate_brewery_type_sk",
        duplicate_brewery_type_sk,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        "dim_location",
        "duplicate_location_sk",
        duplicate_location_sk,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        "dim_brewery",
        "duplicate_brewery_sk",
        duplicate_brewery_sk,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        "fact_brewery",
        "duplicate_fact_records",
        duplicate_fact_records,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        "fact_brewery",
        "null_surrogate_keys",
        null_surrogate_keys,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        "fact_brewery",
        "negative_brewery_count",
        negative_brewery_count,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        "fact_brewery",
        "orphan_brewery_sk",
        orphan_brewery_sk,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        "fact_brewery",
        "orphan_location_sk",
        orphan_location_sk,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        "fact_brewery",
        "orphan_brewery_type_sk",
        orphan_brewery_type_sk,
        datetime.utcnow()
    )
]

dq_df = spark.createDataFrame(
    dq_data,
    dq_schema
)

(
    dq_df.write

    .format("delta")

    .mode("append")

    .saveAsTable(data_quality_table)
)